In [33]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data" / "raw"

orders = pd.read_csv(data_dir / "orders.csv")
order_items = pd.read_csv(data_dir / "order_items.csv")
products = pd.read_csv(data_dir / "products.csv")

print("orders:", orders.shape)
print("order_items:", order_items.shape)
print("products:", products.shape)

orders: (6000, 5)
order_items: (14603, 5)
products: (300, 4)


In [34]:
display(orders.head())
display(order_items.head())
display(products.head())

,order_id,customer_id,order_date,payment_method,order_status
0,1,1121,2026-05-18,간편결제,배송중
1,2,993,2026-02-01,신용카드,환불
2,3,261,2025-05-08,신용카드,배송완료
3,4,783,2026-02-18,신용카드,결제완료
4,5,292,2025-06-20,신용카드,배송완료


,order_item_id,order_id,product_id,quantity,unit_price
0,1,1,216,4,13600
1,2,1,34,1,87000
2,3,1,187,1,93000
3,4,1,112,1,225000
4,5,2,214,1,64000


,product_id,product_name,category,price
0,1,프리미엄 웹캠 그레이 P001,전자기기,68000
1,2,스마트 미니 가습기 블랙 P002,생활가전,163000
2,3,컴팩트 후드 티셔츠 그레이 P003,패션,22000
3,4,프로 세럼 화이트 P004,뷰티,43000
4,5,스마트 드립백 커피 화이트 P005,식품,58000


In [35]:
order_items["item_amount"] = (
    order_items["quantity"] * order_items["unit_price"]
)

In [36]:
display(
    order_items[
        ["quantity", "unit_price", "item_amount"]
    ].head()
)

,quantity,unit_price,item_amount
0,4,13600,54400
1,1,87000,87000
2,1,93000,93000
3,1,225000,225000
4,1,64000,64000


In [37]:
before_rows = len(order_items)

merged = order_items.merge(
    orders,
    on="order_id",
    how="left"
)

print("병합 전 행 수:", before_rows)
print("병합 후 행 수:", len(merged))

병합 전 행 수: 14603
병합 후 행 수: 14603


In [38]:
display(merged.head())

,order_item_id,order_id,product_id,quantity,unit_price,item_amount,customer_id,order_date,payment_method,order_status
0,1,1,216,4,13600,54400,1121,2026-05-18,간편결제,배송중
1,2,1,34,1,87000,87000,1121,2026-05-18,간편결제,배송중
2,3,1,187,1,93000,93000,1121,2026-05-18,간편결제,배송중
3,4,1,112,1,225000,225000,1121,2026-05-18,간편결제,배송중
4,5,2,214,1,64000,64000,993,2026-02-01,신용카드,환불


In [39]:
assert len(merged) == before_rows
assert merged["order_date"].notna().all()
assert merged["order_status"].notna().all()

print("orders 병합 검증 완료")

orders 병합 검증 완료


In [40]:
merged = merged.merge(
    products,
    on="product_id",
    how="left"
)

In [41]:
display(merged.head())

,order_item_id,order_id,product_id,quantity,unit_price,item_amount,customer_id,order_date,payment_method,order_status,product_name,category,price
0,1,1,216,4,13600,54400,1121,2026-05-18,간편결제,배송중,컴팩트 에세이 그린 P216,도서,16000
1,2,1,34,1,87000,87000,1121,2026-05-18,간편결제,배송중,클래식 클렌징 폼 화이트 P034,뷰티,87000
2,3,1,187,1,93000,93000,1121,2026-05-18,간편결제,배송중,데일리 러닝 벨트 그레이 P187,스포츠,93000
3,4,1,112,1,225000,225000,1121,2026-05-18,간편결제,배송중,플러스 핸디 청소기 블루 P112,생활가전,250000
4,5,2,214,1,64000,64000,993,2026-02-01,신용카드,환불,스마트 클렌징 폼 화이트 P214,뷰티,64000


In [42]:
order_items["item_amount"] = (
    order_items["quantity"] * order_items["unit_price"]
)

In [43]:
display(
    order_items[
        ["quantity", "unit_price", "item_amount"]
    ].head()
)

,quantity,unit_price,item_amount
0,4,13600,54400
1,1,87000,87000
2,1,93000,93000
3,1,225000,225000
4,1,64000,64000


In [44]:
sample = order_items.iloc[0]

expected = sample["quantity"] * sample["unit_price"]
actual = sample["item_amount"]

print("예상값:", expected)
print("실제값:", actual)

assert expected == actual

예상값: 54400
실제값: 54400


In [45]:
total_orders = merged["order_id"].nunique()
total_quantity = merged["quantity"].sum()
total_sales = merged["item_amount"].sum()

order_totals = (
    merged.groupby("order_id")["item_amount"]
    .sum()
)

average_order_amount = order_totals.mean()

print("전체 주문 수:", total_orders)
print("총 주문 수량:", total_quantity)
print(f"총 주문 금액: {total_sales:,.0f}원")
print(f"평균 주문 금액: {average_order_amount:,.0f}원")

전체 주문 수: 6000
총 주문 수량: 23596
총 주문 금액: 1,837,774,300원
평균 주문 금액: 306,296원


In [46]:
category_sales = (
    merged
    .groupby("category", as_index=False)["item_amount"]
    .sum()
    .sort_values("item_amount", ascending=False)
)

display(category_sales)

,category,item_amount
4,생활가전,457423100
8,패션,313067200
7,전자기기,286750100
9,홈인테리어,173982200
5,스포츠,154676700
3,뷰티,135823500
6,식품,116157900
2,반려동물,110816200
1,문구,46005800
0,도서,43071600


In [47]:
category_sales_display = category_sales.copy()

category_sales_display["item_amount"] = (
    category_sales_display["item_amount"]
    .map(lambda x: f"{x:,.0f}원")
)

display(category_sales_display)

,category,item_amount
4,생활가전,"457,423,100원"
8,패션,"313,067,200원"
7,전자기기,"286,750,100원"
9,홈인테리어,"173,982,200원"
5,스포츠,"154,676,700원"
3,뷰티,"135,823,500원"
6,식품,"116,157,900원"
2,반려동물,"110,816,200원"
1,문구,"46,005,800원"
0,도서,"43,071,600원"


In [48]:
display(
    category_sales_display.reset_index(drop=True)
)

,category,item_amount
0,생활가전,"457,423,100원"
1,패션,"313,067,200원"
2,전자기기,"286,750,100원"
3,홈인테리어,"173,982,200원"
4,스포츠,"154,676,700원"
5,뷰티,"135,823,500원"
6,식품,"116,157,900원"
7,반려동물,"110,816,200원"
8,문구,"46,005,800원"
9,도서,"43,071,600원"


In [49]:
assert category_sales["item_amount"].sum() == merged["item_amount"].sum()

print("카테고리별 매출 집계 검증 완료")

카테고리별 매출 집계 검증 완료


In [50]:
status_counts = (
    merged[
        ["order_id", "order_status"]
    ]
    .drop_duplicates()
    .groupby("order_status", as_index=False)["order_id"]
    .nunique()
    .rename(columns={"order_id": "order_count"})
    .sort_values("order_count", ascending=False)
)

display(status_counts)

,order_status,order_count
1,배송완료,4862
4,취소,458
5,환불,352
3,배송중,163
2,배송준비,92
0,결제완료,73


In [51]:
assert status_counts["order_count"].sum() == merged["order_id"].nunique()

print("주문 상태별 건수 검증 완료")

주문 상태별 건수 검증 완료


In [52]:
merged["order_date"] = pd.to_datetime(
    merged["order_date"]
)

In [53]:
merged["order_month"] = (
    merged["order_date"]
    .dt.to_period("M")
    .astype(str)
)

In [54]:
monthly_sales = (
    merged
    .groupby("order_month", as_index=False)["item_amount"]
    .sum()
    .sort_values("order_month")
)

display(monthly_sales)

,order_month,item_amount
0,2025-01,87322800
1,2025-02,68504700
2,2025-03,85551000
3,2025-04,104790900
4,2025-05,105969700
5,2025-06,82491800
6,2025-07,86357700
7,2025-08,81312400
8,2025-09,98212700
9,2025-10,84413300


In [55]:
assert monthly_sales["item_amount"].sum() == merged["item_amount"].sum()

print("월별 주문 금액 검증 완료")

월별 주문 금액 검증 완료


In [56]:
import sys

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.analysis import (
    calculate_total_orders,
    calculate_total_quantity,
    calculate_total_sales,
    calculate_average_order_amount,
    get_category_sales,
    get_status_counts,
    get_monthly_sales,
)

In [57]:
print("전체 주문 수:", calculate_total_orders(merged))
print("총 주문 수량:", calculate_total_quantity(merged))
print("총 주문 금액:", calculate_total_sales(merged))
print("평균 주문 금액:", calculate_average_order_amount(merged))

전체 주문 수: 6000
총 주문 수량: 23596
총 주문 금액: 1837774300
평균 주문 금액: 306295.7166666667


In [58]:
assert calculate_total_orders(merged) == total_orders
assert calculate_total_quantity(merged) == total_quantity
assert calculate_total_sales(merged) == total_sales
assert calculate_average_order_amount(merged) == average_order_amount

print("핵심 지표 함수 검증 완료")

핵심 지표 함수 검증 완료


In [59]:
category_result = get_category_sales(merged)
status_result = get_status_counts(merged)
monthly_result = get_monthly_sales(merged)

display(category_result)
display(status_result)
display(monthly_result)

,category,item_amount
4,생활가전,457423100
8,패션,313067200
7,전자기기,286750100
9,홈인테리어,173982200
5,스포츠,154676700
3,뷰티,135823500
6,식품,116157900
2,반려동물,110816200
1,문구,46005800
0,도서,43071600


,order_status,order_count
1,배송완료,4862
4,취소,458
5,환불,352
3,배송중,163
2,배송준비,92
0,결제완료,73


,order_month,item_amount
0,2025-01,87322800
1,2025-02,68504700
2,2025-03,85551000
3,2025-04,104790900
4,2025-05,105969700
5,2025-06,82491800
6,2025-07,86357700
7,2025-08,81312400
8,2025-09,98212700
9,2025-10,84413300


In [60]:
assert category_result["item_amount"].sum() == total_sales
assert status_result["order_count"].sum() == total_orders
assert monthly_result["item_amount"].sum() == total_sales

print("분석 함수 전체 검증 완료")

분석 함수 전체 검증 완료
